# Resume–Job Description Matching with a Shared Siamese BiLSTM

This clean notebook orchestrates the modular project code. The original exploratory notebook is preserved under `archive/`.

## Responsible Use

Educational demonstration only. Use synthetic or anonymized text and do not make employment decisions from the output.

In [ ]:
from pathlib import Path
import sys
PROJECT_DIR = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
PROJECT_DIR

In [ ]:
import pandas as pd
from src.config import CONFIG
from src.pair_generation import generate_balanced_pairs
resumes = pd.read_csv(CONFIG.data_dir / "sample" / "sample_resumes.csv")
jobs = pd.read_csv(CONFIG.data_dir / "sample" / "sample_job_descriptions.csv")
pairs = generate_balanced_pairs(resumes, jobs)
pairs.head()

In [ ]:
pairs.groupby(["split", "label"]).size().unstack(fill_value=0)

## Train the corrected shared encoder

Run this only when you intentionally want to regenerate artifacts. The Streamlit app loads the saved artifacts and does not train at startup.

In [ ]:
# from src.model_training import train_from_pairs
# result = train_from_pairs(pairs)
# result["metrics"]

## Inference

In [ ]:
from src.inference_pipeline import ResumeJobMatcher
matcher = ResumeJobMatcher()
matcher.predict(
    "Data scientist with Python SQL machine learning NLP and Power BI experience.",
    "Seeking a machine learning engineer with Python NLP SQL and model deployment skills.",
)

## Ranking

In [ ]:
from src.ranking_pipeline import rank_resumes
job_text = jobs[jobs["category"] == "Data Science"].iloc[-1]["job_description"]
rank_resumes(job_text, resumes[["resume_id", "resume_text"]].to_dict(orient="records"), matcher=matcher)